In [1]:


import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

SCALE = 6          # sharper than 4 → better separation
THRESH = 0.01      # pruning threshold


# -----------------------------
# Gate Layer (FIXED)
# -----------------------------
class GateLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.w = nn.Parameter(torch.randn(out_features, in_features) * 0.01)
        self.b = nn.Parameter(torch.zeros(out_features))

        # ✅ FIX 1: Proper initialization (NOT -2)
        self.g = nn.Parameter(torch.zeros(out_features, in_features))

    def forward(self, x):
        gate = torch.sigmoid(self.g * SCALE)

        # ✅ FIX 2: prevent regrowth (hard pruning)
        gate = torch.where(gate < THRESH, torch.zeros_like(gate), gate)

        w_eff = self.w * gate
        return F.linear(x, w_eff, self.b)

    def penalty(self):
        return torch.sum(torch.sigmoid(self.g * SCALE))

    def gate_values(self):
        with torch.no_grad():
            return torch.sigmoid(self.g * SCALE)


# -----------------------------
# Model
# -----------------------------
class SparseMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = GateLayer(3072, 800)
        self.l2 = GateLayer(800, 400)
        self.l3 = GateLayer(400, 200)
        self.l4 = GateLayer(200, 10)

        self.bn1 = nn.BatchNorm1d(800)
        self.bn2 = nn.BatchNorm1d(400)
        self.bn3 = nn.BatchNorm1d(200)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = F.relu(self.bn1(self.l1(x)))
        x = F.relu(self.bn2(self.l2(x)))
        x = F.relu(self.bn3(self.l3(x)))
        return self.l4(x)

    def sparsity_term(self):
        total = 0
        for m in self.modules():
            if isinstance(m, GateLayer):
                total += torch.sum(torch.sigmoid(m.g * SCALE))
        return total

    def sparsity_ratio(self):
        total = zero = 0
        for m in self.modules():
            if isinstance(m, GateLayer):
                g = torch.sigmoid(m.g * SCALE)
                total += g.numel()
                zero += (g < THRESH).sum().item()
        return zero / total

    def all_gate_values(self):
        parts = []
        for m in self.modules():
            if isinstance(m, GateLayer):
                parts.append(m.gate_values().cpu().numpy().ravel())
        return np.concatenate(parts)


# -----------------------------
# Data
# -----------------------------
def load_data(batch_size=256):
    mean, std = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)

    train_tf = T.Compose([
        T.RandomHorizontalFlip(),
        T.RandomCrop(32, padding=4),
        T.ToTensor(),
        T.Normalize(mean, std),
    ])

    test_tf = T.Compose([
        T.ToTensor(),
        T.Normalize(mean, std)
    ])

    train_ds = torchvision.datasets.CIFAR10(
        "./data", train=True, download=True, transform=train_tf)

    test_ds = torchvision.datasets.CIFAR10(
        "./data", train=False, download=True, transform=test_tf)

    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True),
        DataLoader(test_ds, batch_size=batch_size, shuffle=False)
    )


# -----------------------------
# Training
# -----------------------------
def train_epoch(net, loader, optimizer, lam, ep, epochs):
    net.train()
    total_loss = correct = total = 0

    # ✅ FIX 3: lambda warm-up
    lam_t = lam * (ep / epochs)

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        out = net(x)

        loss = F.cross_entropy(out, y) + lam_t * net.sparsity_term()

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (out.argmax(1) == y).sum().item()
        total += y.size(0)

    return total_loss / len(loader), correct / total


@torch.no_grad()
def evaluate(net, loader):
    net.eval()
    correct = total = 0

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        correct += (net(x).argmax(1) == y).sum().item()
        total += y.size(0)

    return correct / total


# -----------------------------
# Plot
# -----------------------------
def plot_distributions(all_results):
    fig, axes = plt.subplots(1, len(all_results), figsize=(6 * len(all_results), 4))

    for ax, res in zip(axes, all_results):
        ax.hist(res["gates"], bins=80, range=(0, 1), alpha=0.8)
        ax.axvline(THRESH, linestyle="--", label="threshold")

        ax.set_title(
            f"λ={res['lam']:.0e}\nAcc={res['acc']*100:.1f}%  Sparsity={res['sparsity']*100:.1f}%"
        )
        ax.set_xlabel("Gate value")
        ax.legend()

    plt.tight_layout()
    plt.savefig("gate_distributions_fixed.png")
    print("Saved: gate_distributions_fixed.png")


# -----------------------------
# Main
# -----------------------------
def run():
    train_loader, test_loader = load_data()

    lambdas = [1e-5, 5e-5, 1e-4]
    epochs = 40

    all_results = []

    for lam in lambdas:
        print(f"\n{'='*50}")
        print(f"Lambda = {lam}")
        print(f"{'='*50}")

        model = SparseMLP().to(DEVICE)
        opt = torch.optim.Adam(model.parameters(), lr=1e-3)

        for ep in range(1, epochs + 1):
            loss, tr_acc = train_epoch(model, train_loader, opt, lam, ep, epochs)
            val_acc = evaluate(model, test_loader)
            sparsity = model.sparsity_ratio()

            print(f"Ep {ep:02d} | Loss {loss:.3f} | Train {tr_acc*100:.1f}% | "
                  f"Test {val_acc*100:.1f}% | Sparsity {sparsity*100:.1f}%")

        final_acc = evaluate(model, test_loader)
        final_sparsity = model.sparsity_ratio()

        all_results.append({
            "lam": lam,
            "acc": final_acc,
            "sparsity": final_sparsity,
            "gates": model.all_gate_values(),
        })

    plot_distributions(all_results)


if __name__ == "__main__":
    run()

Device: cpu


100%|██████████| 170M/170M [00:04<00:00, 39.5MB/s]



Lambda = 1e-05
Ep 01 | Loss 2.137 | Train 34.9% | Test 43.8% | Sparsity 0.0%
Ep 02 | Loss 2.163 | Train 43.0% | Test 46.0% | Sparsity 0.0%
Ep 03 | Loss 2.234 | Train 45.9% | Test 50.3% | Sparsity 0.0%
Ep 04 | Loss 2.261 | Train 48.2% | Test 51.1% | Sparsity 0.0%
Ep 05 | Loss 2.264 | Train 49.8% | Test 52.8% | Sparsity 0.0%
Ep 06 | Loss 2.250 | Train 50.9% | Test 53.3% | Sparsity 0.0%
Ep 07 | Loss 2.226 | Train 52.3% | Test 54.9% | Sparsity 0.5%
Ep 08 | Loss 2.198 | Train 53.1% | Test 55.5% | Sparsity 2.3%
Ep 09 | Loss 2.163 | Train 54.5% | Test 56.4% | Sparsity 5.3%
Ep 10 | Loss 2.135 | Train 54.9% | Test 56.4% | Sparsity 9.1%
Ep 11 | Loss 2.099 | Train 55.9% | Test 56.6% | Sparsity 13.4%
Ep 12 | Loss 2.074 | Train 56.6% | Test 57.7% | Sparsity 17.7%
Ep 13 | Loss 2.039 | Train 57.1% | Test 58.7% | Sparsity 22.1%
Ep 14 | Loss 2.011 | Train 57.7% | Test 58.9% | Sparsity 26.2%
Ep 15 | Loss 1.979 | Train 58.7% | Test 59.3% | Sparsity 30.2%
Ep 16 | Loss 1.953 | Train 58.8% | Test 59.7% | S